# India Care Map — Datathon Fall 2026 Submission

**Team Name:** tsampath (andrewID of the submitting member)
**Names:** Tej Sampath, Jeremy Caro, James Ryman
**AndrewIDs:** tsampath, jcaro, jryman
**Submitting member:** tsampath

- **Live app:** https://india-issue-map-7474649106532354.aws.databricksapps.com
- **Source code:** https://github.com/tejxsampath-eng/datathon-fall-2026 (everything lives in `india-issue-map/`)

## What we built

**India Care Map** is a Databricks web app that turns a plain-language medical problem ("helping a pregnant woman near Nashik — her doctor said the delivery could be risky") into the nearest appropriate healthcare facility in India, with a transparent confidence score for every recommendation.

* Patient describes the problem in free text → an AI model (Unity Gateway, `system.ai.gpt-oss-120b`) extracts the **needed medical services** (e.g. obstetrics, NICU, dialysis) and an **urgency level** (emergency / urgent / routine).
* The app geocodes the location (Nominatim, India-restricted, with typo correction), then applies **urgency-based rules** over 9,219 facilities to pick 1–3 recommendations: emergency → nearest 90%-certain hospital; urgent/routine → closest equipped vs. closest certain within the radius.
* Every card shows **why**: which needs are covered, the facility's existence confidence, quality score, per-procedure evidence, and an expandable "How these scores were computed" breakdown in plain language.

## 1. Data

The starting dataset was a 10,000-row India healthcare facility review. Our data pipeline:

1. **Verification audit** — 10,000 source rows were reconciled against 14 evidence files (field changes, review queue, identity/clinical evidence, web research, published-value sources) into **3,240 published facility branches** with cleaned name/address/city/state fields.
2. **Geocoding** — every facility address was geocoded via Nominatim (India-restricted); 675 unique city+state combinations were geocoded and verified (±50 km plausibility check), with deterministic jitter so city-level pins don't stack.
3. **Scoring** — the audit's scoring scripts were reverse-engineered and re-implemented so the app reproduces every stored score exactly (validation in section 4). The final scored dataset ships with the app: `facilities_medical.csv` (17 MB), `facilities_explain.csv.gz` (the scoring payload), and `procedure_lookup.csv`.

## 2. The three scoring models

All three are **deterministic, hand-weighted models** from the dataset's original scoring scripts — no training data, fully reproducible.

### existence_confidence — "is this place real and still open?"
A logistic model over four checks (each min–max scaled between the dataset's 1st–99th percentiles):

`log-odds = −1·recency + 1·has_address + matches_tier/3 + 1·age` → `score = 1/(1+e^−odds)`

* recency: days since last website/social activity, scaled in [1.0, 4357.4] (missing = stalest)
* age: years since establishment, scaled in [3.0, 118.0]
* matches_tier: found in 0–3 outside reference datasets

### quality_score — "how good is this facility?"
Weighted average of 0–1 components, **renormalized over the data each facility type has** (clinics aren't penalized for lacking hospital fields):

| Component | Weight | Source |
| --- | --- | --- |
| NABH accreditation | 35% | empanelled → certified → accredited (0–3) |
| Life-saving services | 20% | confidence it offers ICU/emergency/CT/MRI/dialysis |
| Breadth of specialties | 18% | count vs. other same-type facilities |
| Advanced equipment | 10% | equipment types named in records |
| Staffing | 10% | doctors + beds vs. peers |
| Affiliations | 5% | teaching / government empanelment |
| Patient rating | 2% | star rating in records |

### procedure_confidence — "can they actually do X?"
Evidence-based lookup over the facility's own records, via a synonym table:

* **direct mention** 0.80–0.90 (+0.05 per extra mention, +0.05 if required equipment listed, +0.03 for matching specialty)
* **equipment only** 0.50 (+0.10 specialty bonus)
* **department only** 0.25
* **explicitly negated** → not offered

These constants encode an evidence hierarchy: the facility's own records beat equipment, which beats inference from a department.

## 3. Reconstruct the full app from this notebook

Running the next cell rebuilds the complete `india-issue-map/` folder (Flask backend, Leaflet frontend, config, requirements, datasets) next to this notebook — pulled from the team's GitHub repo, so this single file plus internet is all a grader needs.

## 4. Validation — the ports are faithful

We verified the app-side re-implementations against the dataset's stored scores for **all 9,219 scored facilities**:

| Check | Result |
| --- | --- |
| existence_confidence recreated from the 4 inputs | **exact 9,217/9,219** (2 rows share a name+city key and can't be joined individually) |
| quality_score recreated from the 7 components | exact **8,535** + **682** flagged "approximate" (inputs changed after the stored score was computed — breakdown still shown) |
| procedure evidence levels | logic identical to the original script (direct/supporting/inferred/negated) |

## 5. Live scoring demo

The next cell re-implements the two dataset scorers inline (the exact formulas above), loads the scoring payload that ships with the app, and prints a full recommendation-style breakdown for sample facilities — no Databricks credentials needed.

## 6. Running the app

**Already live:** https://india-issue-map-7474649106532354.aws.databricksapps.com (Databricks Apps; the AI interpreter runs on Unity Gateway `system.ai.gpt-oss-120b`).

**Locally:**
```bash
cd india-issue-map
pip install -r requirements.txt
python app.py        # serves on http://localhost:8080
```
The map, geocoding, typo-correction and recommendations work without any credentials; the AI interpretation and Databricks-table issue lookup use env vars from `app.yaml`.

**Repo structure:** `india-issue-map/` = `app.py` (backend + scoring ports), `templates/index.html` (frontend), `app.yaml`, `requirements.txt`, `facilities_medical.csv`, `facilities_explain.csv.gz`, `procedure_lookup.csv`, `data.csv` (3,240 published facilities on the map).

In [0]:
# Rebuild india-issue-map/ next to this notebook from the team's GitHub repo
import os, urllib.request

BASE = "https://raw.githubusercontent.com/tejxsampath-eng/datathon-fall-2026/main/india-issue-map/"
FILES = ["app.py", "app.yaml", "requirements.txt", "data-template.csv",
         "facilities_medical.csv", "facilities_explain.csv.gz",
         "procedure_lookup.csv", "templates/index.html"]
# data.csv (3,240 published map points) is fetched too but is listed separately
# only if present at push time.
ALL = FILES + ["data.csv"]

for rel in ALL:
    dst = os.path.join("india-issue-map", rel)
    if os.path.exists(dst):
        print("already present:", dst)
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        urllib.request.urlretrieve(BASE + rel, dst)
        print("downloaded:", dst, f"({os.path.getsize(dst):,} bytes)")
    except Exception as e:
        print("skipped:", rel, "—", e)
print("\nDone — india-issue-map/ is ready to run: cd india-issue-map && pip install -r requirements.txt && python app.py")

In [0]:
# Live demo: the two dataset scorers re-implemented inline (the exact
# formulas described above), VALIDATED against the dataset's stored scores
# for every facility, plus sample breakdowns and a procedure-evidence demo.
import csv, gzip, math, os

EXIST_DAYS_RANGE = (1.0, 4357.4)   # 1st–99th pct: days since last online activity
EXIST_YEARS_RANGE = (3.0, 118.0)   # 1st–99th pct: years since establishment
QUALITY_WEIGHTS = {"accreditation": 0.35, "services": 0.20, "specialties": 0.18,
                   "equipment": 0.10, "staffing": 0.10, "rating": 0.02,
                   "affiliation": 0.05}
QUALITY_APPLIES = {"hospital": tuple(QUALITY_WEIGHTS),
                   "clinic": ("accreditation", "specialties", "equipment",
                              "staffing", "rating"),
                   "dentist": ("accreditation", "specialties", "equipment",
                               "staffing", "rating")}
NABH = {0: "no NABH status", 1: "NABH-empanelled", 2: "NABH-certified", 3: "NABH-accredited"}

def _scaled(raw, rng, fill):
    try:
        x = float(raw)
    except (TypeError, ValueError):
        x = fill                      # missing -> the script's chosen default
    x = min(max(x, rng[0]), rng[1])
    return (x - rng[0]) / (rng[1] - rng[0])

def existence_confidence(days_raw, years_raw, addr, tier):
    xd = _scaled(days_raw, EXIST_DAYS_RANGE, EXIST_DAYS_RANGE[1])  # missing = stalest
    xy = _scaled(years_raw, EXIST_YEARS_RANGE, 0.0)
    log_odds = (-xd + (1.0 if str(addr) == "1" else 0.0)
                + min(3, max(0, int(tier or 0))) / 3.0 + xy)
    return 1 / (1 + math.exp(-log_odds))

def quality_score(facility, ftype):
    applies = QUALITY_APPLIES.get(ftype, tuple(QUALITY_WEIGHTS))
    num = den = 0.0
    for name, w in QUALITY_WEIGHTS.items():
        if name not in applies:
            continue
        try:
            v = float(facility.get("q_" + name) or "")
        except ValueError:
            continue                      # no data -> weight renormalized away
        num += v * w; den += w
    return num / den if den else None

BASE = "india-issue-map"
with gzip.open(os.path.join(BASE, "facilities_explain.csv.gz"),
               "rt", newline="", encoding="utf-8") as f:
    payload = {(r["name"], r["city"]): r for r in csv.DictReader(f)}
with open(os.path.join(BASE, "facilities_medical.csv"),
          encoding="utf-8", newline="") as f:
    medical = list(csv.DictReader(f))
print(f"payload rows: {len(payload):,} | scored facilities: {len(medical):,}")

# ---- validation against the dataset's stored scores (all facilities) ----
e_ok = e_tot = q_ok = q_tot = approx = 0
for m in medical:
    p = payload.get((m["name"], m["city"]))
    if not p:
        continue
    e_tot += 1
    mine = existence_confidence(p["days_raw"], p["years_raw"], p["addr"], p["matches_tier"])
    if abs(mine - float(m["existence_confidence"])) <= 0.005:
        e_ok += 1
    q_tot += 1
    mine_q = quality_score(p, m["type"])
    if mine_q is None:
        continue
    if abs(mine_q - float(m["quality_score"])) <= 0.005:
        q_ok += 1
    elif p.get("approx") == "1":
        approx += 1
print(f"existence recreated EXACTLY for {e_ok:,}/{e_tot:,} facilities")
print(f"quality    recreated for {q_ok:,}/{q_tot:,} "
      f"({q_tot - q_ok - approx:,} unexplained, {approx:,} flagged 'approximate' in the data)")

# ---- sample breakdowns ----
def find(prefix=None, ftype=None):
    for m in medical:
        if prefix and not m["name"].startswith(prefix):
            continue
        if ftype and m["type"] != ftype:
            continue
        p = payload.get((m["name"], m["city"]))
        if p:
            return m, p

for label, kw in [("Well-documented hospital", {"prefix": "Medanta"}),
                  ("Clinic (sparse data — weights renormalize over what exists)",
                   {"ftype": "clinic"})]:
    hit = find(**kw)
    if not hit:
        continue
    m, p = hit
    print(f"\n== {label}: {m['name']} ({m['type']}, {m['city']})")
    print("   existence: recomputed %.0f%% vs stored %.0f%%" % (
        existence_confidence(p["days_raw"], p["years_raw"], p["addr"], p["matches_tier"]) * 100,
        float(m["existence_confidence"]) * 100))
    mine_q = quality_score(p, m["type"])
    if mine_q is not None:
        print("   quality  : recomputed %.0f%% vs stored %.0f%%  [%s; services %s; %s specialties]" % (
            mine_q * 100, float(m["quality_score"]) * 100,
            NABH.get(int(p["nabh_grade"] or 0)), p["q_services"] or "n/a",
            p["n_spec"] or "0"))
    else:
        print("   quality  : no applicable components for this facility type")

# ---- procedure evidence (app's constants; the full app adds synonyms) ----
def procedure_confidence(term, p):
    term = term.lower()
    text = " ".join(p.get(c, "") or "" for c in
                    ("procedure", "capability", "description")).lower()
    equip = (p.get("equipment") or "").lower()
    spec = (p.get("specialties") or "").lower()
    if term in text:
        s = 0.80
        if term in equip: s = min(0.90, s + 0.05)
        if term in spec:  s = min(0.90, s + 0.03)
        return s, "direct: named in the place's own records"
    if term in equip:
        return 0.50 + (0.10 if term in spec else 0.0), "supporting: required equipment listed"
    if term in spec:
        return 0.25, "inferred: only a matching department is listed"
    return 0.0, "no evidence in this place's records"

med = find("Medanta")
med = med[1] if med else next(iter(payload.values()))
print("\nprocedure evidence for", med["name"], ":")
for term in ["icu", "dialysis", "ct scan", "cataract surgery"]:
    s, why = procedure_confidence(term, med)
    print(f"   {term:>16}: {s:.2f}  ({why})")